# 📦 01 — Load Data & Pengumpulan Publikasi Dosen
**Skripsi:** Sistem Rekomendasi Dosen Pembimbing Berbasis NLP — Teknik Informatika Unila

---

### Strategi Data (Diperbarui)

| # | Data | Cara Mendapatkan | Status |
|---|------|-----------------|--------|
| 1 | Judul Skripsi + Dosen Pembimbing | **Load CSV** dari sistem monitoring | ✅ Sudah ada |
| 2 | Publikasi Dosen (judul + abstrak) | **Scraping SINTA** per dosen | ⬜ Dikerjakan di notebook ini |

> ⚠️ **Jalankan cell secara berurutan.**

---
## 🔧 LANGKAH 0 — Mount Drive & Install Library

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive ter-mount.')

In [ ]:
!pip install -q requests beautifulsoup4 lxml tqdm
print('✅ Library siap.')

In [ ]:
import sys, os
PROJECT_NAME = 'skripsi-rekomendasi-dosen'  # sesuaikan jika beda
ROOT = f'/content/drive/MyDrive/{PROJECT_NAME}'
sys.path.insert(0, os.path.join(ROOT, 'src'))

import config
import pandas as pd
import numpy as np
import requests, time, re
from bs4 import BeautifulSoup
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f'✅ Konfigurasi dimuat. Root: {ROOT}')

---
## 📚 BAGIAN A — Load Data Skripsi dari CSV

Upload file CSV ke Google Drive terlebih dahulu:
```
MyDrive/skripsi-rekomendasi-dosen/data/raw/skripsi_unila.csv
```
Atau jalankan cell upload di bawah untuk mengunggah langsung dari komputer.

In [ ]:
# ─── OPSI 1: Upload file langsung dari komputer (sekali pakai) ───
# Jalankan cell ini HANYA jika file belum ada di Drive
# Jika sudah ada di Drive, skip ke OPSI 2

from google.colab import files
print('📂 Pilih file CSV dari komputer kamu...')
uploaded = files.upload()

for filename in uploaded.keys():
    # Simpan ke folder raw di Drive
    dest = os.path.join(config.DATA_RAW, 'skripsi_unila.csv')
    with open(dest, 'wb') as f:
        f.write(uploaded[filename])
    print(f'✅ File tersimpan ke: {dest}')

In [ ]:
# ─── OPSI 2: Load langsung dari Drive (jika sudah tersimpan) ─────
csv_path = config.FILE_SKRIPSI_RAW
df_skripsi = pd.read_csv(csv_path)

print('✅ Data skripsi berhasil dimuat!')
print(f'   Jumlah data   : {len(df_skripsi)} baris')
print(f'   Kolom         : {list(df_skripsi.columns)}')

In [ ]:
# ─── EDA: INSPEKSI AWAL ──────────────────────────────────────────
print('=' * 60)
print('📋  RINGKASAN DATA SKRIPSI')
print('=' * 60)
print(f'Total data             : {len(df_skripsi)}')
print(f'Judul kosong           : {df_skripsi["Judul Penelitian"].isna().sum()}')
print(f'Pembimbing kosong      : {df_skripsi["Dosen Pembimbing"].isna().sum()}')
print(f'Dosen unik             : {df_skripsi["Dosen Pembimbing"].nunique()}')
print(f'Topik unik (raw)       : {df_skripsi["Topik Penelitian"].nunique()}')
print()
print('📊  Status Usulan:')
print(df_skripsi['Status Usulan'].value_counts().to_string())
df_skripsi.head()

In [ ]:
# ─── EDA: DISTRIBUSI DOSEN PEMBIMBING ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Grafik 1: Jumlah bimbingan per dosen
dosen_count = df_skripsi['Dosen Pembimbing'].value_counts()
dosen_count.plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Jumlah Bimbingan per Dosen', fontweight='bold')
axes[0].set_xlabel('Jumlah Skripsi')
axes[0].invert_yaxis()

# Grafik 2: Status Usulan
status_count = df_skripsi['Status Usulan'].value_counts()
colors = ['#2196F3','#4CAF50','#FF9800','#9C27B0','#F44336']
status_count.plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                  colors=colors[:len(status_count)], startangle=90)
axes[1].set_title('Distribusi Status Usulan', fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, 'eda_skripsi.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📊 Grafik tersimpan di results/')

In [ ]:
# ─── CLEANING: NORMALISASI KOLOM TOPIK PENELITIAN ────────────────
# Kolom 'Topik Penelitian' sangat berantakan (104 nilai unik).
# Di sini kita normalisasi ke kategori yang lebih bersih.
# Juga kita pisahkan kolom yang relevan untuk NLP.

print('🔤 Topik Penelitian sebelum normalisasi (top 20):')
print(df_skripsi['Topik Penelitian'].value_counts().head(20).to_string())

# Mapping normalisasi topik
TOPIK_MAP = {
    # RPL / Web / Software
    'RPL'                           : 'Rekayasa Perangkat Lunak',
    'Rpl'                           : 'Rekayasa Perangkat Lunak',
    'rpl'                           : 'Rekayasa Perangkat Lunak',
    'Web'                           : 'Rekayasa Perangkat Lunak',
    'Web Development'               : 'Rekayasa Perangkat Lunak',
    'App Development (Backend)'     : 'Rekayasa Perangkat Lunak',
    'Mobile Apps'                   : 'Rekayasa Perangkat Lunak',
    # AI / ML
    'AI'                            : 'Kecerdasan Buatan',
    'Machine Learning'              : 'Kecerdasan Buatan',
    'Deep Learning'                 : 'Kecerdasan Buatan',
    'Computer Vision'               : 'Kecerdasan Buatan',
    'Data Mining'                   : 'Kecerdasan Buatan',
    'Deteksi Anomali'               : 'Kecerdasan Buatan',
    'Data Mining, Machine Learning, Artificial Intelligence': 'Kecerdasan Buatan',
    # Sistem Informasi
    'TI'                            : 'Sistem Informasi',
    'Sistem Informasi'              : 'Sistem Informasi',
    'SISTEM INFORMASI'              : 'Sistem Informasi',
    'Database'                      : 'Sistem Informasi',
    'ERP'                           : 'Sistem Informasi',
    # UI/UX
    'UI/UX'                         : 'UI/UX & Desain',
    'UX'                            : 'UI/UX & Desain',
    # Jaringan
    'Jaringan'                      : 'Jaringan & Keamanan',
    'Network'                       : 'Jaringan & Keamanan',
    'Keamanan Jaringan'             : 'Jaringan & Keamanan',
    'IoT'                           : 'Jaringan & Keamanan',
}

df_skripsi['Topik Normalized'] = df_skripsi['Topik Penelitian'].map(TOPIK_MAP).fillna(df_skripsi['Topik Penelitian'])
print(f'\n✅ Topik setelah normalisasi ({df_skripsi["Topik Normalized"].nunique()} unik):')
print(df_skripsi['Topik Normalized'].value_counts().head(15).to_string())

In [ ]:
# ─── BUAT KOLOM TEKS UTAMA UNTUK NLP ─────────────────────────────
# Kolom ini yang akan dipakai sebagai INPUT ke model TF-IDF / BERT
# Gabungkan Judul + Topik agar lebih representatif

df_skripsi['teks_input'] = (
    df_skripsi['Judul Penelitian'].fillna('') + ' ' +
    df_skripsi['Topik Normalized'].fillna('')
)

# Rename kolom agar konsisten di notebook berikutnya
df_skripsi = df_skripsi.rename(columns={
    'Judul Penelitian' : 'judul',
    'Dosen Pembimbing' : 'pembimbing',
    'Topik Penelitian' : 'topik_raw',
    'Topik Normalized' : 'topik',
    'Nama Mahasiswa'   : 'nama',
    'Tahun Masuk'      : 'tahun_masuk',
})

# Pilih kolom yang relevan
df_skripsi = df_skripsi[['NPM','nama','tahun_masuk','judul','topik_raw','topik','pembimbing','teks_input','Status Usulan','Tanggal Pengajuan']]

print('✅ Kolom teks_input dibuat.')
print(f'   Contoh teks_input[0]:\n   "{df_skripsi["teks_input"].iloc[0]}"')
print(f'\n   Contoh teks_input[1]:\n   "{df_skripsi["teks_input"].iloc[1]}"')

In [ ]:
# ─── SIMPAN DATA SKRIPSI YANG SUDAH BERSIH ───────────────────────
out_path = config.FILE_SKRIPSI_RAW
df_skripsi.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'💾 Data skripsi (cleaned) tersimpan: {out_path}')
print(f'   Jumlah baris : {len(df_skripsi)}')
print(f'   Kolom        : {list(df_skripsi.columns)}')

---
## 👨‍🏫 BAGIAN B — Scraping Publikasi Dosen dari SINTA

14 dosen yang ada di data sudah teridentifikasi. Tinggal cari **SINTA ID** masing-masing.

### Cara cari SINTA ID:
1. Buka [sinta.kemdikbud.go.id](https://sinta.kemdikbud.go.id)
2. Klik **Search** → ketik nama dosen
3. Klik profil dosen → lihat URL: `…/authors/profile/**6661552**`
4. Angka di akhir URL = SINTA ID

> 💡 Jika dosen tidak terdaftar di SINTA, kolom `sinta_id` diisi `None` dan data diambil dari Google Scholar secara manual.

In [ ]:
# ─── DAFTAR 14 DOSEN + SINTA ID ──────────────────────────────────
# ⚠️  ISI sinta_id masing-masing dosen berdasarkan hasil pencarian di SINTA!
# Daftar nama dosen diambil otomatis dari data CSV.

DOSEN_LIST = [
    {'nama': 'Dr. Eng. Ir. Mardiana, S.T., M.T., IPM.',       'sinta_id': None, 'bidang': ''},
    {'nama': 'Dr. Ir. M. Komarudin, M.T.',                    'sinta_id': None, 'bidang': ''},
    {'nama': 'Ir. Gigih Forda Nama, S.T., M.T.I, IPM.',       'sinta_id': None, 'bidang': ''},
    {'nama': 'Ir. Ing. Hery Dian Septama, S.T., IPM.',        'sinta_id': None, 'bidang': ''},
    {'nama': 'Ir. Meizano Ardhi Muhammad, S.T., M.T., IPM.',  'sinta_id': None, 'bidang': ''},
    {'nama': 'Ir. Resty Annisa, S.ST., M.Kom.',               'sinta_id': None, 'bidang': ''},
    {'nama': 'Ir. Titin Yulianti, S.T., M.Eng.',              'sinta_id': None, 'bidang': ''},
    {'nama': 'Ir. Trisya Septiana, ST.,MT., IPM.',            'sinta_id': None, 'bidang': ''},
    {'nama': 'Mahendra Pratama, S.T., M.Eng.',                'sinta_id': None, 'bidang': ''},
    {'nama': 'Mona Arif Muda, M.T.',                          'sinta_id': None, 'bidang': ''},
    {'nama': 'Puput Budi Wintoro, S. Kom, M.T.I',             'sinta_id': None, 'bidang': ''},
    {'nama': 'Rio Ariestia Pradipta, S.Kom.,M.T.I.',          'sinta_id': None, 'bidang': ''},
    {'nama': 'Wahyu Eko Sulistiono, M.Sc.',                   'sinta_id': None, 'bidang': ''},
    {'nama': 'Yessi Mulyani, M.T.',                           'sinta_id': None, 'bidang': ''},
]

print(f'📋 {len(DOSEN_LIST)} dosen terdaftar.')
print('⚠️  Isi kolom sinta_id dan bidang sebelum lanjut ke scraping!')

# Cek berapa yang sudah diisi
terisi = sum(1 for d in DOSEN_LIST if d['sinta_id'] is not None)
print(f'   Sudah ada SINTA ID : {terisi}/{len(DOSEN_LIST)}')

In [ ]:
# ─── FUNGSI SCRAPING SINTA ───────────────────────────────────────
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                  'AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36'
}
DELAY = 2.0  # detik antar request

def scrape_sinta(sinta_id, nama_dosen, max_page=5):
    """Ambil daftar publikasi dosen dari SINTA berdasarkan SINTA ID."""
    results = []
    for page in range(1, max_page + 1):
        url = (
            f'https://sinta.kemdikbud.go.id/authors/detail'
            f'?id={sinta_id}&view=googlescholar&page={page}'
        )
        try:
            r    = requests.get(url, headers=HEADERS, timeout=15)
            soup = BeautifulSoup(r.text, 'lxml')
            items = soup.select('div.ar-list-item')
            if not items:
                break
            for item in items:
                title_tag = item.select_one('div.ar-title a') or item.select_one('a')
                judul = title_tag.get_text(strip=True) if title_tag else ''
                raw   = item.get_text()
                tahun = (re.search(r'\b(20\d{2}|19\d{2})\b', raw) or type('', (), {'group': lambda s,x: ''})()).group(0) if re.search(r'\b(20\d{2}|19\d{2})\b', raw) else ''
                venue_tag = item.select_one('div.ar-source')
                venue = venue_tag.get_text(strip=True) if venue_tag else ''
                if judul:
                    results.append({
                        'nama_dosen' : nama_dosen,
                        'sinta_id'   : sinta_id,
                        'judul_pub'  : judul,
                        'tahun'      : tahun,
                        'venue'      : venue,
                    })
            time.sleep(DELAY)
        except Exception as e:
            print(f'  ⚠️  Error {nama_dosen} page {page}: {e}')
            break
    return results

print('✅ Fungsi scraping SINTA siap.')

In [ ]:
# ─── JALANKAN SCRAPING ────────────────────────────────────────────
# ⚠️  PASTIKAN sinta_id di DOSEN_LIST sudah diisi!

dosen_dengan_sinta = [d for d in DOSEN_LIST if d['sinta_id'] is not None]
dosen_tanpa_sinta  = [d for d in DOSEN_LIST if d['sinta_id'] is None]

print(f'🚀 Scraping publikasi untuk {len(dosen_dengan_sinta)} dosen (yang ada SINTA ID)...')
if dosen_tanpa_sinta:
    print(f'⚠️  {len(dosen_tanpa_sinta)} dosen tanpa SINTA ID (perlu input manual nanti):')
    for d in dosen_tanpa_sinta:
        print(f'   - {d["nama"]}')
print()

all_pub = []
for dosen in tqdm(dosen_dengan_sinta):
    nama = dosen['nama']
    print(f'  👤 {nama[:40]}... ', end='')
    pubs = scrape_sinta(dosen['sinta_id'], nama, max_page=5)
    all_pub.extend(pubs)
    print(f'{len(pubs)} publikasi')
    time.sleep(DELAY)

df_dosen = pd.DataFrame(all_pub) if all_pub else pd.DataFrame(
    columns=['nama_dosen','sinta_id','judul_pub','tahun','venue']
)
print(f'\n📊 Total publikasi: {len(df_dosen)}')

In [ ]:
# ─── INPUT MANUAL untuk dosen tanpa SINTA ────────────────────────
# Jika ada dosen yang tidak terdaftar di SINTA,
# tambahkan publikasi secara manual di sini.
# Format: list of dict dengan key: nama_dosen, judul_pub, tahun, venue

PUBLIKASI_MANUAL = [
    # Contoh (hapus dan ganti jika ada):
    # {
    #     'nama_dosen' : 'Nama Dosen X',
    #     'sinta_id'   : 'manual',
    #     'judul_pub'  : 'Judul Penelitian Dosen X Tahun 2023',
    #     'tahun'      : '2023',
    #     'venue'      : 'Jurnal/Prosiding',
    # },
]

if PUBLIKASI_MANUAL:
    df_manual = pd.DataFrame(PUBLIKASI_MANUAL)
    df_dosen  = pd.concat([df_dosen, df_manual], ignore_index=True)
    print(f'✅ {len(PUBLIKASI_MANUAL)} publikasi manual ditambahkan.')
else:
    print('ℹ️  Tidak ada input manual.')

print(f'📊 Total publikasi keseluruhan: {len(df_dosen)}')

In [ ]:
# ─── BUAT PROFIL TEKS PER DOSEN ──────────────────────────────────
# Gabungkan semua judul publikasi dosen menjadi satu 'dokumen profil'
# Ini yang akan dijadikan representasi dosen di sistem rekomendasi

if not df_dosen.empty:
    profil_dosen = (
        df_dosen.groupby('nama_dosen')['judul_pub']
        .apply(lambda x: ' '.join(x.dropna().tolist()))
        .reset_index()
        .rename(columns={'judul_pub': 'profil_teks'})
    )

    # Tambahkan bidang keahlian dari DOSEN_LIST
    bidang_map = {d['nama']: d.get('bidang', '') for d in DOSEN_LIST}
    profil_dosen['bidang'] = profil_dosen['nama_dosen'].map(bidang_map)
    profil_dosen['jumlah_pub'] = df_dosen.groupby('nama_dosen').size().values

    print('✅ Profil teks dosen berhasil dibuat:')
    print(profil_dosen[['nama_dosen','jumlah_pub','bidang']].to_string())
    print(f'\nContoh profil_teks dosen pertama (100 karakter pertama):')
    print(f'  "{profil_dosen["profil_teks"].iloc[0][:100]}..."')
else:
    print('⚠️  Data publikasi masih kosong — isi sinta_id di DOSEN_LIST terlebih dahulu.')
    profil_dosen = pd.DataFrame(columns=['nama_dosen','profil_teks','bidang','jumlah_pub'])

In [ ]:
# ─── SIMPAN SEMUA DATA ───────────────────────────────────────────

# 1. Data publikasi mentah
df_dosen.to_csv(config.FILE_DOSEN_RAW, index=False, encoding='utf-8-sig')
print(f'💾 publikasi_dosen.csv → {config.FILE_DOSEN_RAW}')

# 2. Profil dosen (1 baris per dosen)
profil_path = os.path.join(config.DATA_RAW, 'profil_dosen.csv')
profil_dosen.to_csv(profil_path, index=False, encoding='utf-8-sig')
print(f'💾 profil_dosen.csv     → {profil_path}')

print()
print('📁 File yang tersimpan di data/raw/ :')
for f in os.listdir(config.DATA_RAW):
    fpath = os.path.join(config.DATA_RAW, f)
    size  = os.path.getsize(fpath) / 1024
    print(f'   {f:<35} ({size:.1f} KB)')

---
## ✅ Selesai — Ringkasan Notebook 01

| File | Isi | Status |
|------|-----|--------|
| `data/raw/skripsi_unila.csv` | 277 skripsi + kolom `teks_input` | ✅ |
| `data/raw/publikasi_dosen.csv` | Semua publikasi mentah per dosen | ✅ |
| `data/raw/profil_dosen.csv` | 1 baris per dosen + `profil_teks` gabungan | ✅ |
| `results/eda_skripsi.png` | Grafik distribusi dosen & status | ✅ |

### ⚠️ Yang Masih Perlu Dilakukan:
- **Isi `sinta_id`** di `DOSEN_LIST` untuk semua 14 dosen
- Untuk dosen yang tidak ada di SINTA → tambahkan di `PUBLIKASI_MANUAL`

### 🗺️ Langkah Berikutnya:
> **`02_preprocessing.ipynb`** — Cleaning teks, stopword removal, stemming Bahasa Indonesia